# RabTech Academy Task 2 — KPI Dictionary & Data Quality Contract

**Domain:** Retail commerce orders  
**Decision owner:** Retail Operations Manager  
**Data owner:** Data Product Owner / Data Engineering  
**Purpose:** Profile raw order data and run repeatable data-quality checks.

## Run instructions
1. Download `retail-orders-raw.csv` from the official task page.
2. Put it in the same folder as this notebook (or update `DATA_PATH` below).
3. Run all cells top-to-bottom.
4. Review `data_quality_results.csv` and `data_profile_summary.csv`. The sample may fail freshness because it is historical; that is intentional and should be escalated according to the contract.

The notebook discovers common column-name variants. Review the printed column mapping and adapt it to the official data dictionary if necessary.


In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from datetime import datetime, timezone

DATA_PATH = Path("retail-orders-raw.csv")
if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    print("CSV files found:", [p.name for p in candidates])
    raise FileNotFoundError(
        "Place the official raw file named 'retail-orders-raw.csv' beside this notebook, "
        "then rerun this cell. Download it from the RabTech Academy Task 2 page."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows × {len(df.columns):,} columns")
display(df.head())
print("Columns:", list(df.columns))


In [ ]:
def norm_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def find_col(candidates):
    lookup = {norm_name(c): c for c in df.columns}
    for candidate in candidates:
        if norm_name(candidate) in lookup:
            return lookup[norm_name(candidate)]
    # allow common variants by containment
    for c in df.columns:
        nc = norm_name(c)
        for candidate in candidates:
            ncan = norm_name(candidate)
            if ncan and (ncan in nc or nc in ncan):
                return c
    return None

COLS = {
    "order_id": find_col(["order_id", "orderid", "invoice", "invoice_no", "transaction_id"]),
    "order_date": find_col(["order_date", "orderdate", "date", "created_at", "order_timestamp"]),
    "customer_id": find_col(["customer_id", "customerid", "client_id", "buyer_id"]),
    "product_id": find_col(["product_id", "productid", "sku", "item_id", "stock_code"]),
    "quantity": find_col(["quantity", "qty", "units"]),
    "unit_price": find_col(["unit_price", "unitprice", "price", "price_each"]),
    "sales": find_col(["sales", "revenue", "line_total", "total_amount", "amount"]),
    "status": find_col(["status", "order_status", "fulfillment_status"]),
}
print("Detected column mapping:")
for k, v in COLS.items():
    print(f"  {k}: {v}")


In [ ]:
# Build reusable profiling metrics and test results.
results = []
def add_check(check, dimension, failed_count, denominator, threshold_pct, severity, action, details=""):
    denominator = int(denominator) if denominator is not None else 0
    failed_count = int(failed_count) if failed_count is not None else 0
    rate = (100 * failed_count / denominator) if denominator else 0.0
    passed = rate <= threshold_pct
    results.append({
        "check": check, "dimension": dimension, "failed_count": failed_count,
        "row_count": denominator, "failure_rate_pct": round(rate, 4),
        "threshold_pct": threshold_pct, "severity": severity,
        "status": "PASS" if passed else "FAIL", "action": action, "details": details
    })

n = len(df)
# Completeness: critical fields required for basic order analysis.
critical_fields = [x for x in [COLS["order_id"], COLS["order_date"], COLS["quantity"]] if x]
for col in critical_fields:
    missing = df[col].isna() | df[col].astype(str).str.strip().eq("")
    add_check(f"Completeness — {col} non-null", "completeness", missing.sum(), n, 1.0, "High",
              f"Quarantine affected rows; notify Data Owner if threshold exceeded.", f"Column={col}")

# Uniqueness: duplicated candidate line key.
key_cols = [x for x in [COLS["order_id"], COLS["product_id"]] if x]
if len(key_cols) >= 2:
    dup_mask = df.duplicated(subset=key_cols, keep=False)
    add_check("Uniqueness — candidate order/product key", "uniqueness", dup_mask.sum(), n, 0.5, "High",
              "Inspect repeated lines; deduplicate only after confirming line-item semantics.",
              f"Candidate key={key_cols}; duplicate rows are flagged, not auto-deleted.")
elif COLS["order_id"]:
    dup_mask = df.duplicated(subset=[COLS["order_id"]], keep=False)
    add_check("Uniqueness — order ID duplicate diagnostic", "uniqueness", dup_mask.sum(), n, 100.0, "Medium",
              "Order IDs may legitimately repeat across line items; review before treating as a failure.",
              "Order ID alone is not necessarily a unique row key.")
else:
    add_check("Uniqueness — no key available", "uniqueness", n, n, 0.0, "High",
              "Identify a stable order-line key in the data dictionary.", "No order identifier detected.")

# Validity: quantities should be numeric and positive.
if COLS["quantity"]:
    q = pd.to_numeric(df[COLS["quantity"]], errors="coerce")
    invalid = q.isna() | (q <= 0)
    add_check(f"Validity — {COLS['quantity']} numeric and > 0", "validity", invalid.sum(), n, 0.5, "High",
              "Quarantine rows with nonnumeric or non-positive quantity; confirm returns/refunds policy.",
              "If negative quantities encode returns, define a separate approved rule.")
# Validity: unit price should be numeric and non-negative.
if COLS["unit_price"]:
    p = pd.to_numeric(df[COLS["unit_price"]], errors="coerce")
    invalid = p.isna() | (p < 0)
    add_check(f"Validity — {COLS['unit_price']} numeric and >= 0", "validity", invalid.sum(), n, 0.5, "High",
              "Quarantine invalid prices and investigate source-system mapping.")
# Date parse validity.
if COLS["order_date"]:
    parsed_dates = pd.to_datetime(df[COLS["order_date"]], errors="coerce", dayfirst=False)
    invalid = parsed_dates.isna()
    add_check(f"Validity — {COLS['order_date']} parseable date", "validity", invalid.sum(), n, 0.5, "High",
              "Quarantine unparseable dates; confirm source format and locale.")
    future = parsed_dates > pd.Timestamp.now().normalize() + pd.Timedelta(days=1)
    add_check(f"Consistency — {COLS['order_date']} not in future", "consistency", future.sum(), n, 0.1, "Medium",
              "Check timezone, source clock, and date parsing; correct only from source evidence.")
else:
    parsed_dates = None

# Consistency: if a sales/amount column and quantity + unit price exist, compare line total.
if COLS["sales"] and COLS["quantity"] and COLS["unit_price"]:
    qty = pd.to_numeric(df[COLS["quantity"]], errors="coerce")
    price = pd.to_numeric(df[COLS["unit_price"]], errors="coerce")
    sales = pd.to_numeric(df[COLS["sales"]], errors="coerce")
    expected = qty * price
    comparable = expected.notna() & sales.notna()
    inconsistent = comparable & ~np.isclose(expected, sales, rtol=0.02, atol=0.02)
    add_check("Consistency — sales ~= quantity × unit price", "consistency", inconsistent.sum(), comparable.sum(), 1.0, "Medium",
              "Review tax, discounts, currency, and rounding rules before correcting totals.",
              f"Compared {int(comparable.sum())} rows; tolerance 2% or 0.02 units.")
else:
    print("Sales consistency check skipped: need sales, quantity and unit-price columns.")

# Freshness: latest order date should be no more than 3 days old for daily refresh.
if COLS["order_date"] and parsed_dates is not None and parsed_dates.notna().any():
    latest = parsed_dates.max()
    age_days = (pd.Timestamp.now().normalize() - latest.normalize()).days
    results.append({
        "check": "Freshness — latest order date <= 3 days old",
        "dimension": "freshness", "failed_count": int(age_days > 3), "row_count": n,
        "failure_rate_pct": 100.0 if age_days > 3 else 0.0, "threshold_pct": 0.0,
        "severity": "High", "status": "PASS" if age_days <= 3 else "FAIL",
        "action": "If a production feed is expected, alert Data Engineering and Retail Operations; for a historical starter file, document that it is a static sample.",
        "details": f"Latest date={latest}; age_days={age_days}"
    })
else:
    results.append({
        "check": "Freshness — order date available", "dimension": "freshness",
        "failed_count": 1, "row_count": 1, "failure_rate_pct": 100.0, "threshold_pct": 0.0,
        "severity": "High", "status": "FAIL",
        "action": "Confirm the date column mapping with the supplied data dictionary.",
        "details": "No parseable order-date column detected."
    })

quality = pd.DataFrame(results)
display(quality)
quality.to_csv("data_quality_results.csv", index=False)
print("Saved data_quality_results.csv")


In [ ]:
# Dataset-level profile: completeness, uniqueness, validity, consistency, and freshness.
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "row_count": [len(df)] * len(df.columns),
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [round(100 * df[c].isna().mean(), 3) for c in df.columns],
    "distinct_count": [int(df[c].nunique(dropna=True)) for c in df.columns],
    "distinct_pct": [round(100 * df[c].nunique(dropna=True) / max(len(df), 1), 3) for c in df.columns],
    "sample_values": [", ".join(df[c].dropna().astype(str).head(3).tolist())[:150] for c in df.columns]
})
display(profile)
profile.to_csv("data_profile_summary.csv", index=False)
print("Saved data_profile_summary.csv")

print("\nOverall status:", "FAIL — review failed checks" if (quality["status"] == "FAIL").any() else "PASS — all defined checks passed")
print("Important: a failure is a signal to investigate; do not silently delete or rewrite raw data.")
